# Ejecución Inicial del Pipeline OKSP  
## Generación de Detecciones y Picks utilizando EQTransformer


## Carga de Bibliotecas Principales

In [1]:
import argparse
import csv
import json
import os
import shutil
import sys

from datetime import datetime, timedelta
from glob import glob
from os import (
    listdir,
    makedirs,
    path,
    symlink,
    system,
)

import warnings


import pandas as pd
from obspy import read

import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))


2026-05-15 12:39:42.044233: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcudart.so.10.1


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


2026-05-15 12:39:42.660059: I tensorflow/compiler/jit/xla_cpu_device.cc:41] Not creating XLA devices, tf_xla_enable_xla_devices not set
2026-05-15 12:39:42.660867: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcuda.so.1
2026-05-15 12:39:42.787723: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:941] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2026-05-15 12:39:42.792420: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1720] Found device 0 with properties: 
pciBusID: 0000:01:00.0 name: NVIDIA GeForce RTX 4050 Laptop GPU computeCapability: 8.9
coreClock: 1.605GHz coreCount: 20 deviceMemorySize: 5.64GiB deviceMemoryBandwidth: 178.84GiB/s
2026-05-15 12:39:42.792482: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcudart.so.10.1
2026-05-15 12:39:42.793588: I tensorflow/stream_executor/

## Data Transfer

In [2]:
base_dir = "/home/sebas/Downloads/Workshop_IA/Workshop_Results/2026/133"


if os.path.exists(base_dir):
    shutil.rmtree(base_dir)

os.makedirs(base_dir)

waves_in_dir = "/home/sebas/Downloads/Workshop_IA/Data"

startdate = "2026-133"
enddate   = "2026-133"

master_station_file = "/home/sebas/Downloads/Workshop_IA/stations/stations.csv"

CONT_FOLDER = 'CONT_WAVES'

DB_MSEED_PATH_FMT = ('{waves_in_dir}/{year}/{julday}/'
                     'i4.{station}.{channel}.{year}{julday}_0+'
)

STA_FILEPATH = "{base_dir}/stations.csv"

## Crear stations.csv
Limitado a las estaciones que vamos a trabajar, se extrae desde las waves disponibles.

In [3]:
year, day = startdate.split("-")

files = glob(
    os.path.join(
        waves_in_dir,
        year,
        day,
        "*"
    )
)

stations_found = set(
    os.path.basename(f).split(".")[1]
    for f in files
)

output_station_dir = os.path.join(base_dir, "stations")
os.makedirs(output_station_dir, exist_ok=True)

output_station_file = os.path.join(
    output_station_dir,
    "stations.csv"
)

with open(master_station_file, 'r') as infile, \
     open(output_station_file, 'w', newline='') as outfile:

    reader = csv.DictReader(infile)
    writer = csv.DictWriter(
        outfile,
        fieldnames=reader.fieldnames
    )

    writer.writeheader()

    for row in reader:

        if row['code'] in stations_found:
            writer.writerow(row)

shutil.copy2(
    output_station_file,
    os.path.join(base_dir, 'stations.csv')
)

print(f"Stations file created:")
print(output_station_file)

Stations file created:
/home/sebas/Downloads/Workshop_IA/Workshop_Results/2026/133/stations/stations.csv


## COPIAR WAVEFORMS

In [4]:
df = pd.read_csv(
    STA_FILEPATH.format(base_dir=base_dir)
)

start = datetime.strptime(startdate, '%Y-%j')
end   = datetime.strptime(enddate,   '%Y-%j')

for day in range(int((end - start).days) + 1):

    tt = (start + timedelta(day)).timetuple()

    year = tt.tm_year
    yday = tt.tm_yday

    year_path = path.join(
        base_dir,
        CONT_FOLDER,
        str(year)
    )

    yday_path = path.join(
        year_path,
        f'{yday}'
    )

    for folder in [year_path, yday_path]:

        if not path.exists(folder):
            makedirs(folder)

    print(f"\nProcesando dia: {year}-{yday}")

    for _, row in df.iterrows():

        station = row.code
        channels = row.channels.split('_')

        for channel in channels:

            filepath = DB_MSEED_PATH_FMT.format(
                waves_in_dir = waves_in_dir,
                year         = year,
                julday       = yday,
                station      = station,
                channel      = channel
            )

            cmd = f'scp {filepath} {yday_path}/.'

            print(cmd)

            system(cmd)


Procesando dia: 2026-133
scp /home/sebas/Downloads/Workshop_IA/Data/2026/133/i4.CDM.HHE.2026133_0+ /home/sebas/Downloads/Workshop_IA/Workshop_Results/2026/133/CONT_WAVES/2026/133/.
scp /home/sebas/Downloads/Workshop_IA/Data/2026/133/i4.CDM.HHN.2026133_0+ /home/sebas/Downloads/Workshop_IA/Workshop_Results/2026/133/CONT_WAVES/2026/133/.
scp /home/sebas/Downloads/Workshop_IA/Data/2026/133/i4.CDM.HHZ.2026133_0+ /home/sebas/Downloads/Workshop_IA/Workshop_Results/2026/133/CONT_WAVES/2026/133/.
scp /home/sebas/Downloads/Workshop_IA/Data/2026/133/i4.PEZE.HHE.2026133_0+ /home/sebas/Downloads/Workshop_IA/Workshop_Results/2026/133/CONT_WAVES/2026/133/.
scp /home/sebas/Downloads/Workshop_IA/Data/2026/133/i4.PEZE.HHN.2026133_0+ /home/sebas/Downloads/Workshop_IA/Workshop_Results/2026/133/CONT_WAVES/2026/133/.
scp /home/sebas/Downloads/Workshop_IA/Data/2026/133/i4.PEZE.HHZ.2026133_0+ /home/sebas/Downloads/Workshop_IA/Workshop_Results/2026/133/CONT_WAVES/2026/133/.
scp /home/sebas/Downloads/Workshop_

## Ajuste de trazas

En ocasiones hay trazas que tienen segundos de diferencias, es importante que todos sean exactamente iguales para no alterar la etapa de preprocessing de EQT

In [5]:
base_path = path.join(
    base_dir,
    CONT_FOLDER,
    str(year),
    str(yday)
)

files = glob(
    os.path.join(base_path, "i4.*")
)

stations = set(
    os.path.basename(f).split(".")[1]
    for f in files
)

for sta in stations:

    print(f"\nProcessing station: {sta}")

    sta_pattern = os.path.join(
        base_path,
        f"i4.{sta}.*"
    )

    sta_files = glob(sta_pattern)

    st = read(sta_pattern)

    # Merge
    st.merge(fill_value=0)

    # Tiempo comun
    start_common = max(
        tr.stats.starttime
        for tr in st
    )

    end_common = min(
        tr.stats.endtime
        for tr in st
    )

    st.trim(
        starttime=start_common,
        endtime=end_common
    )

    st.sort()

    file_map = {}

    for f in sta_files:

        cha = os.path.basename(f).split(".")[2]

        file_map[cha] = f


    for tr in st:

        cha = tr.stats.channel

        if cha in file_map:

            tr.write(
                file_map[cha],
                format="MSEED"
            )

        else:
            print(
                f"No match for channel "
                f"{cha} at station {sta}"
            )


Processing station: RIMA

Processing station: OCM

Processing station: QPSB

Processing station: TBLN

Processing station: DMCL

Processing station: OCHAL

Processing station: LAFE

Processing station: SRBA

Processing station: CALV

Processing station: PEZE

Processing station: JACO

Processing station: COLV

Processing station: SASA

Processing station: CDM



## VERIFICACION

Este no es necesario pero es bueno para saber que tenemos.


In [6]:
files = glob(
    os.path.join(base_path, "i4.*")
)

print(f"Total archivos: {len(files)}")

for f in sorted(files[:10]):
    print(os.path.basename(f))

Total archivos: 42
i4.CALV.HHE.2026133_0+
i4.CDM.HHN.2026133_0+
i4.COLV.HHE.2026133_0+
i4.JACO.HHN.2026133_0+
i4.LAFE.HHZ.2026133_0+
i4.OCHAL.HHZ.2026133_0+
i4.OCM.HHN.2026133_0+
i4.RIMA.HHE.2026133_0+
i4.SRBA.HHE.2026133_0+
i4.SRBA.HHN.2026133_0+


## Etapa DB2EQT

En este punto, ya se cuenta con un directorio que contiene los archivos continuos del experimento.  

Sin embargo, estos archivos todavía no cumplen con el formato requerido por EQTransformer, por lo que es necesario realizar una etapa de conversión y reorganización antes de ejecutar el proceso de detección.

In [7]:
DATE_FMT = '%Y-%j'

startdate = "2026-133"
enddate   = "2026-133"

EQT_FOLDER = "DETECTION"
EQT_WAVES_DIR = "WAVES"
EQT_JSON_FNAME = "stations.json"

# Input waveform format
DB_MSEED_PATH_FMT = (
    "{waves_in_dir}/{year}/{julday}/"
    "i4.{station}.{channel}"
)

# EQTransformer filename format
EQT_FNAME_FMT = (
    "{network}.{station}.{location}.{channel}"
    "__{ys:04d}{ms:02d}{ds:02d}T{Hs:02d}{Ms:02d}{Ss:02d}Z"
    "__{ye:04d}{me:02d}{de:02d}T{He:02d}{Me:02d}{Se:02d}Z.mseed"
)

## Validacion del csv de stations que se creo al inicio

In [8]:
df = pd.read_csv(
    STA_FILEPATH.format(base_dir=base_dir)
)

print(df.head())

    code  longitude  latitude  elevation     channels
0    CDM   -83.7637    9.5537     3494.0  HHE_HHN_HHZ
1   PEZE   -83.6775    9.3826      807.0  HHE_HHN_HHZ
2   JACO   -84.6595    9.6624       85.0  HHE_HHN_HHZ
3  OCHAL   -83.6456    9.0976      137.0  HHE_HHN_HHZ
4   RIMA   -83.8636    9.7666     1665.0  HHE_HHN_HHZ


## Creación del Directorio de Trabajo para EQTransformer

En esta etapa se crea el directorio principal donde se almacenará todo el contenido relacionado con el procesamiento mediante EQTransformer.

Inicialmente, este directorio contendrá únicamente los enlaces simbólicos hacia los archivos continuos en la carpeta (`WAVES`) correspondientes al periodo y estaciones seleccionadas.

In [9]:
waves_out_dir = path.join(
    base_dir,
    EQT_FOLDER,
    EQT_WAVES_DIR
)

stations = df.code.tolist()

if not path.exists(waves_out_dir):
    makedirs(waves_out_dir)

for station in stations:

    station_dir = path.join(
        waves_out_dir,
        station
    )

    if not path.exists(station_dir):
        makedirs(station_dir)

print("Folder tree created.")

Folder tree created.


## Crear SYMLINKS

In [10]:
start_dt = datetime.strptime(startdate, DATE_FMT)
end_dt   = datetime.strptime(enddate,   DATE_FMT)

for station in stations:

    print(f"\nProcessing station: {station}")

    channels = (
        df[df.code == station]
        .channels.values[0]
        .split('_')
    )

    for channel in channels:

        for day in range(int((end_dt - start_dt).days) + 1):

            tt = (
                start_dt + timedelta(day)
            ).timetuple()


            in_filepath = DB_MSEED_PATH_FMT.format(
                waves_in_dir = waves_in_dir,
                year         = tt.tm_year,
                julday       = tt.tm_yday,
                station      = station,
                channel      = channel
            )

            files = glob(in_filepath + "*")

            if len(files) == 0:

                print(
                    f"Missing file: "
                    f"{in_filepath}*"
                )

                continue


            for real_file in files:

                print(f"\nFound file:")
                print(real_file)

                try:

                    st = read(
                        real_file,
                        headonly=True
                    )

                    starttime = min(
                        tr.stats.starttime
                        for tr in st
                    )

                    endtime = max(
                        tr.stats.endtime + tr.stats.delta
                        for tr in st
                    )

                    out_filename = EQT_FNAME_FMT.format(

                        network  = st[0].stats.network,
                        station  = st[0].stats.station,
                        location = st[0].stats.location,
                        channel  = st[0].stats.channel,

                        ys = starttime.year,
                        ms = starttime.month,
                        ds = starttime.day,

                        Hs = starttime.hour,
                        Ms = starttime.minute,
                        Ss = starttime.second,

                        ye = endtime.year,
                        me = endtime.month,
                        de = endtime.day,

                        He = endtime.hour,
                        Me = endtime.minute,
                        Se = endtime.second
                    )

                    out_filepath = path.join(
                        waves_out_dir,
                        station,
                        out_filename
                    )


                    os.makedirs(
                        path.dirname(out_filepath),
                        exist_ok=True
                    )


                    if not path.lexists(out_filepath):

                        print(
                            f"Creating symlink:\n"
                            f"{out_filepath}"
                        )

                        symlink(
                            real_file,
                            out_filepath
                        )

                    else:

                        print(
                            f"Already exists:\n"
                            f"{out_filepath}"
                        )

                except Exception as e:

                    print(
                        f"\nError processing "
                        f"{real_file}"
                    )

                    print(e)


Processing station: CDM

Found file:
/home/sebas/Downloads/Workshop_IA/Data/2026/133/i4.CDM.HHE.2026133_0+
Creating symlink:
/home/sebas/Downloads/Workshop_IA/Workshop_Results/2026/133/DETECTION/WAVES/CDM/OV.CDM..HHE__20260513T155959Z__20260513T210000Z.mseed

Found file:
/home/sebas/Downloads/Workshop_IA/Data/2026/133/i4.CDM.HHN.2026133_0+
Creating symlink:
/home/sebas/Downloads/Workshop_IA/Workshop_Results/2026/133/DETECTION/WAVES/CDM/OV.CDM..HHN__20260513T155959Z__20260513T210000Z.mseed

Found file:
/home/sebas/Downloads/Workshop_IA/Data/2026/133/i4.CDM.HHZ.2026133_0+
Creating symlink:
/home/sebas/Downloads/Workshop_IA/Workshop_Results/2026/133/DETECTION/WAVES/CDM/OV.CDM..HHZ__20260513T155959Z__20260513T210000Z.mseed

Processing station: PEZE

Found file:
/home/sebas/Downloads/Workshop_IA/Data/2026/133/i4.PEZE.HHE.2026133_0+
Creating symlink:
/home/sebas/Downloads/Workshop_IA/Workshop_Results/2026/133/DETECTION/WAVES/PEZE/OV.PEZE..HHE__20260513T155959Z__20260513T210000Z.mseed

Found

## Crear stations.json

In [11]:
data = {}

for station in listdir(waves_out_dir):

    network  = set()
    channels = set()

    station_dir = path.join(
        waves_out_dir,
        station
    )

    fnames = listdir(station_dir)

    if len(fnames) == 0:
        continue

    for fname in fnames:

        network.add(
            fname.split('.')[0]
        )

        channels.add(
            fname.split('.')[3].split('_')[0]
        )

    latitude = (
        df[df.code == station]
        .latitude.values[0]
    )

    longitude = (
        df[df.code == station]
        .longitude.values[0]
    )

    elevation = (
        df[df.code == station]
        .elevation.values[0]
    )

    if len(network) > 1:

        warnings.warn(
            f'Several networks for station '
            f'{station}: {network}'
        )

    network = list(network)[0]

    data[station] = {

        'network': network,

        'channels': list(channels),

        'coords': [
            latitude,
            longitude,
            elevation
        ]
    }

json_path = path.join(
    base_dir,
    EQT_FOLDER,
    EQT_JSON_FNAME
)

with open(json_path, 'w') as f:

    json.dump(
        data,
        f,
        indent=4
    )

print(f"\nJSON file created:")
print(json_path)


JSON file created:
/home/sebas/Downloads/Workshop_IA/Workshop_Results/2026/133/DETECTION/stations.json


## Preprocesamiento de Datos para EQTransformer

Antes de ejecutar la detección sísmica, los archivos deben convertirse al formato hdf5 utilizado por EQTransformer.

Durante esta etapa:

- Se leen los archivos MiniSEED continuos
- Se generan archivos HDF5 utilizados por EQTransformer
- Se crean estructuras optimizadas para inferencia mediante redes neuronales
- Se aplica segmentación temporal usando ventanas móviles con traslape (`overlap`)

Además, se utiliza el archivo `stations.json` para asociar correctamente la información de estaciones y componentes.

In [12]:
eqt_path = "/home/sebas/Downloads/Workshop_IA/EQTransformer"


sys.path.insert(0, eqt_path)

from EQTransformer.utils.hdf5_maker import preprocessor


preprocessor(

    preproc_dir = path.join(
        base_dir,
        EQT_FOLDER,
        "preproc"
    ),

    mseed_dir = path.join(
        base_dir,
        EQT_FOLDER,
        EQT_WAVES_DIR
    ),

 
    stations_json = path.join(
        base_dir,
        EQT_FOLDER,
        "stations.json"
    ),

    overlap = 0.9,

    n_processor = 8
)

============ Station JACO has 1 chunks of data.
============ Station PEZE has 1 chunks of data.
============ Station CDM has 1 chunks of data.
============ Station SASA has 1 chunks of data.
============ Station RIMA has 1 chunks of data.
============ Station OCHAL has 1 chunks of data.
============ Station DMCL has 1 chunks of data.
============ Station COLV has 1 chunks of data.
============ Station OCM has 1 chunks of data.
============ Station TBLN has 1 chunks of data.
============ Station CALV has 1 chunks of data.
============ Station QPSB has 1 chunks of data.
============ Station LAFE has 1 chunks of data.
============ Station SRBA has 1 chunks of data.
  * QPSB (1) .. 20260513 --> 20260513 .. 3 components .. sampling rate: 100.0
  * SASA (1) .. 20260513 --> 20260513 .. 3 components .. sampling rate: 100.0
  * COLV (1) .. 20260513 --> 20260513 .. 3 components .. sampling rate: 100.0
  * SRBA (1) .. 20260513 --> 20260513 .. 3 components .. sampling rate: 100.0  * CDM (1) .. 202

## Detección Sísmica utilizando EQTransformer

En esta etapa se ejecuta el modelo de detección y picking de EQTransformer sobre los datos previamente preprocesados.

- Detecciones de eventos sísmicos
- Picks de fases `P`
- Picks de fases `S`
- Probabilidades asociadas a cada detección y pick
- Estimaciones de incertidumbre

Además, se utilizan umbrales de probabilidad para controlar qué tan estricta será la detección automática.

In [13]:
DETECTION_THRESHOLD = 0.60
P_THRESHOLD         = 0.50
S_THRESHOLD         = 0.10


BATCH_SIZE      = 256
NUMBER_OF_CPUS  = 1
GPU_ID          = 0

eqt_path = "/home/sebas/Downloads/Workshop_IA/EQTransformer"
sys.path.insert(0, eqt_path)

from EQTransformer.core.predictor import predictor

predictor(

    input_dir = path.join(
        base_dir,
        EQT_FOLDER,
        EQT_WAVES_DIR + "_processed_hdfs"
    ),

    input_model = "/home/sebas/Downloads/Workshop_IA/final_model.h5",

    output_dir = path.join(
        base_dir,
        EQT_FOLDER,
        "RESULTS"
    ),

    # Estimacion de incertidumbre
    estimate_uncertainty = True,

    # Guardar probabilidades completas
    output_probabilities = False,

    # Multiprocessing
    use_multiprocessing = False,

    # Numero de muestreos Monte Carlo
    number_of_sampling = 3,

    # Pesos de la funcion de perdida
    loss_weights = [0.02, 0.40, 0.58],

    # Umbral de deteccion de eventos
    detection_threshold = DETECTION_THRESHOLD,

    # Umbral para picks P
    P_threshold = P_THRESHOLD,

    # Umbral para picks S
    S_threshold = S_THRESHOLD,

    # Numero de figuras generadas
    number_of_plots = 0,

    # Tamaño de batch
    batch_size = BATCH_SIZE,

    # Numero de CPUs
    number_of_cpus = NUMBER_OF_CPUS,

    # Mantener picks P y S asociados
    keepPS = True,

    # Permitir eventos con solo fase S
    allowonlyS = False,

    # Tiempo maximo S-P permitido (segundos)
    spLimit = 60,

    # GPU utilizada
    gpuid = None,

    # Limite de memoria GPU
    gpu_limit = None
)

Running EqTransformer  0.1.61
 *** Loading the model ...


2026-05-15 12:43:26.068032: I tensorflow/core/platform/cpu_feature_guard.cc:142] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-15 12:43:26.068740: I tensorflow/compiler/jit/xla_gpu_device.cc:99] Not creating XLA devices, tf_xla_enable_xla_devices not set
2026-05-15 12:43:26.069166: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:941] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2026-05-15 12:43:26.072842: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1720] Found device 0 with properties: 
pciBusID: 0000:01:00.0 name: NVIDIA GeForce RTX 4050 Laptop GPU computeCapability: 8.9
coreClock: 1.605GHz coreCount: 20 deviceMemorySize: 5.64GiB deviceMemoryBandwid

Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: module 'gast' has no attribute 'Index'
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: module 'gast' has no attribute 'Index'
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
*** Loading is complete!
######### There are files for 14 stations in /home/sebas/Downloads/Workshop_IA/Workshop_Results/2026/133/DETECTION/WAVES_processed_hdfs directory. #########
========= Started working on CALV, 1 out of 14 ...
  0%|                                                                        | 0/12 [00:00<?, ?it/s]

2026-05-15 12:43:28.723446: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:116] None of the MLIR optimization passes are enabled (registered 2)
2026-05-15 12:43:28.723708: I tensorflow/core/platform/profile_utils/cpu_utils.cc:112] CPU Frequency: 2995200000 Hz
2026-05-15 12:43:31.713946: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcublas.so.10
2026-05-15 12:43:31.984705: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcudnn.so.7


## Unificar outputs

In [ ]:
import glob
output_dir = path.join(
        base_dir,
        EQT_FOLDER
    )

csv_files = glob.glob(
    os.path.join(output_dir,
        "RESULTS", "*_outputs", "X_prediction_results.csv")
)

print(f"Found {len(csv_files)} files")

dfs = [pd.read_csv(f) for f in csv_files]

detecciones = pd.concat(dfs, ignore_index=True)

output_file = os.path.join(output_dir, "detecciones.csv")

detecciones.to_csv(output_file, index=False)

print(f"\nMerged file saved to:\n{output_file}")

print(f"\nTotal rows: {len(detecciones)}")